# RNN HIV 5-Fold Statistics Revize
This notebook evaluates a SimpleRNN classifier with stratified 5-fold CV and exports fold-level and summary statistics.

In [14]:
!pip install rdkit
!pip install torch-geometric
!pip install tensorflow


In [16]:

import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Masking
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")

SEED = 23
MAX_LEN = 32
EMBED_DIM = 64
RNN_UNITS = 64
DROPOUT = 0.40
LR = 5e-4
BATCH_SIZE = 64
EPOCHS = 40
PATIENCE = 8

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)


In [17]:

csv_path = keras.utils.get_file(
    "HIV7.csv",
    "https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv"
)
df = pd.read_csv(csv_path)
df["smiles"] = df["smiles"].astype(str)
df = df[df["smiles"].str.len() < MAX_LEN].reset_index(drop=True)
df = df[["smiles", "HIV_active"]].dropna().reset_index(drop=True)

print(df.shape)
print(df["HIV_active"].value_counts())
df.head()


In [18]:

def build_vocab(smiles_series):
    chars = sorted(set("".join(smiles_series.tolist())))
    char_to_int = {c: i + 1 for i, c in enumerate(chars)}  # 0 reserved for padding
    int_to_char = {i: c for c, i in char_to_int.items()}
    return chars, char_to_int, int_to_char

def vectorize(smiles_series, char_to_int, max_len=MAX_LEN):
    arr = np.zeros((len(smiles_series), max_len), dtype=np.int32)
    for i, s in enumerate(smiles_series.tolist()):
        for j, ch in enumerate(s[:max_len]):
            arr[i, j] = char_to_int.get(ch, 0)
    return arr

def build_rnn_model(vocab_size, max_len=MAX_LEN):
    model = Sequential([
        Embedding(input_dim=vocab_size + 1, output_dim=EMBED_DIM, input_length=max_len, mask_zero=True),
        SimpleRNN(RNN_UNITS, dropout=DROPOUT, recurrent_dropout=0.0),
        Dropout(DROPOUT),
        Dense(64, activation="relu"),
        Dropout(0.20),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                 tf.keras.metrics.AUC(name="auc")]
    )
    return model

def find_best_threshold_by_f1(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.02)
    best_thr = 0.50
    best_f1 = -1.0
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thr = thr
    return float(best_thr), float(best_f1)

def summarize_history(history):
    hist = history.history
    return {
        "loss": hist.get("loss", []),
        "val_loss": hist.get("val_loss", []),
        "accuracy": hist.get("accuracy", hist.get("binary_accuracy", [])),
        "val_accuracy": hist.get("val_accuracy", hist.get("val_binary_accuracy", [])),
        "auc": hist.get("auc", []),
        "val_auc": hist.get("val_auc", [])
    }


In [19]:

def run_fold(train_df, val_df, test_df, fold_id):
    set_seed(SEED + fold_id)

    _, char_to_int, _ = build_vocab(train_df["smiles"])
    vocab_size = len(char_to_int)

    X_train = vectorize(train_df["smiles"], char_to_int, MAX_LEN)
    X_val   = vectorize(val_df["smiles"], char_to_int, MAX_LEN)
    X_test  = vectorize(test_df["smiles"], char_to_int, MAX_LEN)

    y_train = train_df["HIV_active"].astype(np.float32).values
    y_val   = val_df["HIV_active"].astype(np.float32).values
    y_test  = test_df["HIV_active"].astype(np.float32).values

    model = build_rnn_model(vocab_size, MAX_LEN)

    early_stop = EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0,
        shuffle=True
    )

    val_probs = model.predict(X_val, verbose=0).ravel()
    best_threshold, best_val_f1 = find_best_threshold_by_f1(y_val, val_probs)

    test_probs = model.predict(X_test, verbose=0).ravel()
    y_pred = (test_probs >= best_threshold).astype(int)

    fold_result = {
        "fold": fold_id,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred, zero_division=0),
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, test_probs),
        "best_threshold": best_threshold,
        "val_best_f1": best_val_f1,
        "epochs_ran": len(history.history["loss"])
    }

    return fold_result, summarize_history(history)


In [20]:

all_fold_results = []
all_histories = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_index = np.arange(len(df))
y = df["HIV_active"].values

for fold_id, (train_idx, temp_idx) in enumerate(skf.split(X_index, y), start=1):
    train_df = df.iloc[train_idx].reset_index(drop=True)
    temp_df = df.iloc[temp_idx].reset_index(drop=True)

    val_idx, test_idx = train_test_split(
        np.arange(len(temp_df)),
        test_size=0.5,
        random_state=SEED + fold_id,
        stratify=temp_df["HIV_active"]
    )

    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)

    print(f"\n===== Fold {fold_id} =====")
    print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

    fold_result, hist = run_fold(train_df, val_df, test_df, fold_id)
    print(fold_result)

    all_fold_results.append(fold_result)
    all_histories.append(hist)

results_df = pd.DataFrame(all_fold_results)
results_df


In [21]:

summary_rows = []
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    mean = results_df[metric].mean()
    std = results_df[metric].std(ddof=1)
    var = results_df[metric].var(ddof=1)
    summary_rows.append({
        "Metric": metric.replace("test_", "").upper(),
        "Mean": mean,
        "Std": std,
        "Variance": var,
        "Formatted": f"{mean:.3f} ± {std:.3f}"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [22]:

final_table = pd.DataFrame([{
    "Model": "RNN",
    "Accuracy": summary_df.loc[summary_df["Metric"] == "ACCURACY", "Formatted"].iloc[0],
    "Precision": summary_df.loc[summary_df["Metric"] == "PRECISION", "Formatted"].iloc[0],
    "Recall": summary_df.loc[summary_df["Metric"] == "RECALL", "Formatted"].iloc[0],
    "F1": summary_df.loc[summary_df["Metric"] == "F1", "Formatted"].iloc[0],
    "ROC-AUC": summary_df.loc[summary_df["Metric"] == "ROC_AUC", "Formatted"].iloc[0],
}])

results_df.to_csv("RNN_fold_results.csv", index=False)
summary_df.to_csv("RNN_summary_results.csv", index=False)
final_table.to_csv("RNN_final_table.csv", index=False)

final_table


In [23]:

max_len_hist = max(len(h["loss"]) for h in all_histories)

def pad_hist(hist_list, key):
    arr = []
    for h in hist_list:
        vals = h.get(key, [])
        if len(vals) == 0:
            vals = [0.0] * max_len_hist
        elif len(vals) < max_len_hist:
            vals = vals + [vals[-1]] * (max_len_hist - len(vals))
        arr.append(vals)
    return np.array(arr)

def smooth_curve(values, window=5):
    smoothed = []
    for i in range(len(values)):
        start = max(0, i - window + 1)
        smoothed.append(np.mean(values[start:i+1]))
    return smoothed

loss_arr = pad_hist(all_histories, "loss")
val_loss_arr = pad_hist(all_histories, "val_loss")
acc_arr = pad_hist(all_histories, "accuracy")
val_acc_arr = pad_hist(all_histories, "val_accuracy")

plt.figure(figsize=(6,4))
plt.plot(smooth_curve(loss_arr.mean(axis=0), window=5), label="Train Loss")
plt.plot(smooth_curve(val_loss_arr.mean(axis=0), window=5), label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RNN Average 5-Fold Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(smooth_curve(acc_arr.mean(axis=0), window=5), label="Train Accuracy")
plt.plot(smooth_curve(val_acc_arr.mean(axis=0), window=5), label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("RNN Average 5-Fold Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## RNN docking preparation block

This block is prepared for Colab and is as fault-tolerant as possible.

Added workflow:
- RDKit installation cell
- select the best fold
- rebuild the same split
- retrain the model
- top 10 candidates
- shared 13 columns
- final 2 candidates
- colored 2D molecule drawing
- `.smi` docking file

In [24]:
# Colab RDKit install
import sys, subprocess, pkgutil

if pkgutil.find_loader("rdkit") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"])

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

print("RDKit OK")

In [11]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Draw
    from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED
    print("RDKit ready")
except Exception as e:
    print("RDKit yok:", e)

In [25]:
# ==========================================
# RNN FINAL PIPELINE (NO encode_smiles ERROR)
# ==========================================

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras.callbacks import EarlyStopping

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

# 0) seed
seed_value = SEED if "SEED" in globals() else 42
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

# 1) best fold
auc_candidates = ["test_roc_auc", "roc_auc", "val_roc_auc", "auc", "val_auc", "test_auc"]
auc_col = None
if "results_df" in globals():
    for c in auc_candidates:
        if c in results_df.columns:
            auc_col = c
            break

if auc_col is not None:
    best_fold_idx = int(results_df[auc_col].astype(float).idxmax())
else:
    best_fold_idx = 0

best_fold_number = best_fold_idx + 1
print(f"Using best fold: {best_fold_number}")

# 2) veri kontrol
if "df" not in globals():
    raise ValueError("df not found.")
if "smiles" not in df.columns:
    raise ValueError("Column 'smiles' not found in df.")
if "HIV_active" not in df.columns:
    raise ValueError("Column 'HIV_active' not found in df.")
if "build_rnn_model" not in globals():
    raise ValueError("build_rnn_model function not found.")

labels = df["HIV_active"].astype(int).values

# 3) split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_value)
indices = np.arange(len(df))
splits = list(skf.split(indices, labels))

train_idx, temp_idx = splits[best_fold_idx]
train_idx = np.array(train_idx)
temp_idx = np.array(temp_idx)

temp_labels = labels[temp_idx]
val_sub_idx, test_sub_idx = train_test_split(
    np.arange(len(temp_idx)),
    test_size=0.5,
    random_state=seed_value + best_fold_number,
    stratify=temp_labels
)

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[temp_idx[val_sub_idx]].reset_index(drop=True)
test_df  = df.iloc[temp_idx[test_sub_idx]].reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# 4) encoding fallback
def build_char_encoder(smiles_series):
    all_chars = sorted(set("".join(smiles_series.astype(str).tolist())))
    char_to_idx = {ch: i + 1 for i, ch in enumerate(all_chars)}  # 0 padding
    max_len = max(smiles_series.astype(str).map(len))
    return char_to_idx, max_len

def encode_smiles_fallback(smiles_series, char_to_idx=None, max_len=None):
    smiles_series = smiles_series.astype(str).reset_index(drop=True)
    if char_to_idx is None or max_len is None:
        char_to_idx, max_len = build_char_encoder(smiles_series)
    X = np.zeros((len(smiles_series), max_len), dtype=np.int32)
    for i, sm in enumerate(smiles_series):
        seq = [char_to_idx.get(ch, 0) for ch in sm[:max_len]]
        X[i, :len(seq)] = seq
    return X, char_to_idx, max_len

# use the existing encode function if available, otherwise fallback
if "encode_smiles" in globals():
    try:
        X_train = encode_smiles(train_df["smiles"])
        X_val   = encode_smiles(val_df["smiles"])
        X_test  = encode_smiles(test_df["smiles"])
    except Exception:
        X_train, char_to_idx, max_len = encode_smiles_fallback(train_df["smiles"])
        X_val, _, _ = encode_smiles_fallback(val_df["smiles"], char_to_idx, max_len)
        X_test, _, _ = encode_smiles_fallback(test_df["smiles"], char_to_idx, max_len)
else:
    X_train, char_to_idx, max_len = encode_smiles_fallback(train_df["smiles"])
    X_val, _, _ = encode_smiles_fallback(val_df["smiles"], char_to_idx, max_len)
    X_test, _, _ = encode_smiles_fallback(test_df["smiles"], char_to_idx, max_len)

y_train = train_df["HIV_active"].astype(np.float32).values
y_val   = val_df["HIV_active"].astype(np.float32).values
y_test  = test_df["HIV_active"].astype(np.float32).values
smiles_test = test_df["smiles"].reset_index(drop=True)

# 5) hiperparametre fallback
epochs_value = EPOCHS if "EPOCHS" in globals() else 20
batch_size_value = BATCH_SIZE if "BATCH_SIZE" in globals() else 64
patience_value = PATIENCE if "PATIENCE" in globals() else 5

# 6) model kur
try:
    model = build_rnn_model()
except TypeError:
    try:
        model = build_rnn_model(X_train.shape[1])
    except TypeError:
        try:
            model = build_rnn_model(input_length=X_train.shape[1])
        except TypeError:
            try:
                model = build_rnn_model(X_train.shape[1], int(np.max(X_train)) + 1)
            except TypeError:
                raise ValueError("build_rnn_model was found but could not be initialized with suitable parameters.")

monitor_metric = "val_loss"
monitor_mode = "min"
try:
    if any("auc" in m.lower() for m in model.metrics_names):
        monitor_metric = "val_auc"
        monitor_mode = "max"
except Exception:
    pass

early_stop = EarlyStopping(
    monitor=monitor_metric,
    mode=monitor_mode,
    patience=patience_value,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs_value,
    batch_size=batch_size_value,
    callbacks=[early_stop],
    shuffle=True,
    verbose=0
)

# 7) prediction
y_prob = model.predict(X_test, verbose=0).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# 8) df_pred
df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 9) top 10
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 10) descriptors
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

# 11) shared 13 columns
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nRNN TOP 10 CANDIDATES:")
display(df_desc)

# 12) final 2
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nRNN FINAL 2 CANDIDATES:")
display(final_df)

# 13) save
df_desc.to_csv("RNN_top_10_candidates.csv", index=False)
final_df.to_csv("RNN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("RNN_docking_input.smi", index=False, header=False)

print("\nSaved: RNN_top_10_candidates.csv")
print("Saved: RNN_final_2_candidates.csv")
print("Saved: RNN_docking_input.smi")

# 14) colored drawing
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(340, 340),
    legends=[
        f"RNN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"RNN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)

In [26]:
# ==========================================
# RNN FINAL PIPELINE (SAFE SIMPLE VERSION)
# ==========================================

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
import matplotlib.pyplot as plt
import re

# 0) seed
seed_value = SEED if "SEED" in globals() else 42
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

# 1) best fold
auc_candidates = ["test_roc_auc", "roc_auc", "val_roc_auc", "auc", "val_auc", "test_auc"]
auc_col = None
if "results_df" in globals():
    for c in auc_candidates:
        if c in results_df.columns:
            auc_col = c
            break

if auc_col is not None:
    best_fold_idx = int(results_df[auc_col].astype(float).idxmax())
else:
    best_fold_idx = 0

best_fold_number = best_fold_idx + 1
print(f"Using best fold: {best_fold_number}")

# 2) veri kontrol
if "df" not in globals():
    raise ValueError("df not found.")
if "smiles" not in df.columns:
    raise ValueError("Column 'smiles' not found in df.")
if "HIV_active" not in df.columns:
    raise ValueError("Column 'HIV_active' not found in df.")

labels = df["HIV_active"].astype(int).values

# 3) split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_value)
indices = np.arange(len(df))
splits = list(skf.split(indices, labels))

train_idx, temp_idx = splits[best_fold_idx]
train_idx = np.array(train_idx)
temp_idx = np.array(temp_idx)

temp_labels = labels[temp_idx]
val_sub_idx, test_sub_idx = train_test_split(
    np.arange(len(temp_idx)),
    test_size=0.5,
    random_state=seed_value + best_fold_number,
    stratify=temp_labels
)

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[temp_idx[val_sub_idx]].reset_index(drop=True)
test_df  = df.iloc[temp_idx[test_sub_idx]].reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# 4) char-level encoding
def build_char_encoder(smiles_series):
    all_chars = sorted(set("".join(smiles_series.astype(str).tolist())))
    char_to_idx = {ch: i + 1 for i, ch in enumerate(all_chars)}  # 0 padding
    max_len = max(smiles_series.astype(str).map(len))
    return char_to_idx, max_len

def encode_smiles(smiles_series, char_to_idx=None, max_len=None):
    smiles_series = smiles_series.astype(str).reset_index(drop=True)
    if char_to_idx is None or max_len is None:
        char_to_idx, max_len = build_char_encoder(smiles_series)
    X = np.zeros((len(smiles_series), max_len), dtype=np.int32)
    for i, sm in enumerate(smiles_series):
        seq = [char_to_idx.get(ch, 0) for ch in sm[:max_len]]
        X[i, :len(seq)] = seq
    return X, char_to_idx, max_len

X_train, char_to_idx, max_len = encode_smiles(train_df["smiles"])
X_val, _, _ = encode_smiles(val_df["smiles"], char_to_idx, max_len)
X_test, _, _ = encode_smiles(test_df["smiles"], char_to_idx, max_len)

y_train = train_df["HIV_active"].astype(np.float32).values
y_val   = val_df["HIV_active"].astype(np.float32).values
y_test  = test_df["HIV_active"].astype(np.float32).values
smiles_test = test_df["smiles"].reset_index(drop=True)

vocab_size = len(char_to_idx) + 1

# 5) sade ve uyumlu RNN modeli
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    SimpleRNN(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=True,
    verbose=0
)

# 6) prediction
y_prob = model.predict(X_test, verbose=0).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# 7) df_pred
df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 8) top 10
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 9) RDKit'siz descriptor fallback
def compute_desc_no_rdkit(smiles):
    if not isinstance(smiles, str):
        return None

    atoms = re.findall(r'[A-Z][a-z]?', smiles)
    mw_table = {
        'C': 12.011, 'H': 1.008, 'O': 15.999, 'N': 14.007,
        'S': 32.06, 'P': 30.974, 'F': 18.998, 'Cl': 35.45,
        'Br': 79.904, 'I': 126.90, 'Se': 78.971
    }

    mw = sum(mw_table.get(a, 0) for a in atoms)
    hbd = smiles.count('N') + smiles.count('O')
    hba = hbd
    logp = smiles.count('C') * 0.54 - smiles.count('O') * 1.5 - smiles.count('N') * 1.0
    tpsa = hba * 12.0
    rot = smiles.count('(')

    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    qed = min(1.0, max(0.0, 0.6 * (1 / (1 + abs(logp))) + 0.4 * (1 / (1 + hbd))))

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

rows = []
for _, row in top_df.iterrows():
    d = compute_desc_no_rdkit(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

# 10) shared 13 columns
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nRNN TOP 10 CANDIDATES:")
display(df_desc)

# 11) final 2
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nRNN FINAL 2 CANDIDATES:")
display(final_df)

# 12) save
df_desc.to_csv("RNN_top_10_candidates.csv", index=False)
final_df.to_csv("RNN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("RNN_docking_input.smi", index=False, header=False)

print("\nSaved: RNN_top_10_candidates.csv")
print("Saved: RNN_final_2_candidates.csv")
print("Saved: RNN_docking_input.smi")

# 13) visualization
fig, axes = plt.subplots(1, min(2, len(final_df)), figsize=(12, 3))
if len(final_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, final_df.iterrows()):
    ax.axis("off")
    ax.text(
        0.02, 0.5,
        f"SMILES:\n{row['smiles']}\n\nProb={row['y_prob']:.3f}",
        fontsize=10,
        va="center",
        wrap=True,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="black")
    )

plt.tight_layout()
plt.show()

In [27]:
from rdkit import Chem
from rdkit.Chem import Draw

# draw the best 2 molecules in final_df
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"][:2]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(350, 350),
    legends=[
        f"RNN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}",
        f"RNN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}"
    ]
)

display(img)